# Notebook 02: ArXiv Search

This notebook searches ArXiv for papers using the expanded keywords from Notebook 01.

**Input:** `data/keywords_expanded.csv` (from Notebook 01)
**Output:** `data/papers.csv`

**Process:**
1. Load expanded keywords from previous step
2. Query ArXiv API with combined search terms
3. Parse XML responses into structured paper data
4. Apply date range and category filters
5. Save results to CSV with paper metadata

In [ ]:
# Test package availability
import sys
import subprocess

def test_package(package_name, import_name=None):
    if import_name is None:
        import_name = package_name.replace('-', '_')
    try:
        __import__(import_name)
        print(f"✅ {package_name} available")
        return True
    except ImportError:
        print(f"❌ {package_name} missing - please run: uv sync")
        return False

# Test required packages
packages_ok = all([
    test_package('PyYAML', 'yaml'),
    test_package('openai'),
    test_package('pandas'),
    test_package('python-dotenv', 'dotenv'),
    test_package('tqdm'),
    test_package('arxiv')  # New package for ArXiv API
])

if not packages_ok:
    print("\nPlease install missing packages and restart the notebook.")
    sys.exit(1)

In [ ]:
# Import required libraries
import os
import sys
import pandas as pd
import time
from tqdm import tqdm
import arxiv
from datetime import datetime, timedelta

# Set project root directory (works from notebooks directory)
PROJECT_ROOT = os.path.dirname(os.getcwd())
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

# Add virtual environment packages to path
venv_path = os.path.join(PROJECT_ROOT, '.venv', 'lib', 'python3.12', 'site-packages')
sys.path.insert(0, venv_path)

# Add scripts directory to path
scripts_path = os.path.join(PROJECT_ROOT, 'scripts')
sys.path.append(scripts_path)

# Import our utilities
from utils import (
    load_config, 
    get_openai_client, 
    save_csv_checkpoint
)

print("✅ Imports successful")

In [ ]:
# Load configuration
config = load_config()
print(f"Domain: {config['domain']['name']}")
print(f"ArXiv Categories: {config['arxiv']['categories']}")
print(f"Date from: {config['arxiv']['date_from']}")
print(f"Max results: {config['arxiv']['max_results']}")
print(f"Sort by: {config['arxiv']['sort_by']}")

In [ ]:
# Load expanded keywords from previous step
keywords_df = pd.read_csv('data/keywords_expanded.csv')
print(f"Loaded {len(keywords_df)} expanded keywords")
print(f"Categories: {keywords_df['category'].unique()}")

# Show sample
print("\nSample keywords:")
print(keywords_df.head(10))

In [ ]:
# Prepare ArXiv search query
def build_arxiv_query(keywords_df, categories, date_from=None, use_expanded=True, include_date=True):
    """Build ArXiv search query from keywords and categories"""
    
    if use_expanded:
        # Get all expanded keywords
        all_keywords = keywords_df['expanded'].unique().tolist()
    else:
        # Use only original keywords for simpler queries
        all_keywords = keywords_df['original'].unique().tolist()
    
    # Create OR conditions for keywords
    keyword_conditions = [f'all:"{kw}"' for kw in all_keywords]
    keyword_query = ' OR '.join(keyword_conditions)
    
    # Add category filters
    category_conditions = [f'cat:{cat}' for cat in categories]
    category_query = ' OR '.join(category_conditions)
    
    # Start building the query
    query_parts = [f'({keyword_query})', f'({category_query})']
    
    # Add date filter if specified and requested
    if include_date and date_from:
        # Convert date string to YYYYMMDD format for ArXiv API
        if isinstance(date_from, str):
            date_obj = datetime.fromisoformat(date_from)
        else:
            date_obj = date_from
        date_str = date_obj.strftime('%Y%m%d')
        # Filter for papers submitted on or after this date (YYYYMMDDHHMM format)
        # Use a far future date for open-ended range
        future_date = '203001010000'  # January 1, 2030
        date_query = f'submittedDate:[{date_str}0000 TO {future_date}]'
        query_parts.append(f'({date_query})')
    
    # Combine with AND
    full_query = ' AND '.join(query_parts)
    
    return full_query, all_keywords

query, search_keywords = build_arxiv_query(keywords_df, config['arxiv']['categories'], config['arxiv'].get('date_from'), use_expanded=False, include_date=True)
print(f"Search query: {query[:200]}...")
print(f"Using {len(search_keywords)} unique search terms (original keywords only for API compatibility)")

In [ ]:
# Configure ArXiv client
client = arxiv.Client(
    page_size=100,  # Results per page
    delay_seconds=3,  # Rate limiting: 3 seconds between requests
    num_retries=3
)

# Set up search parameters
search = arxiv.Search(
    query=query,
    max_results=config['arxiv']['max_results'],
    sort_by=arxiv.SortCriterion.SubmittedDate,
    sort_order=arxiv.SortOrder.Descending
)

print("✅ ArXiv search configured")
print(f"Max results: {config['arxiv']['max_results']}")
print(f"Date filter: {config['arxiv'].get('date_from', 'None')}")

In [ ]:
# Execute ArXiv search
print("Searching ArXiv... (this may take a few minutes)")

papers_data = []
try:
    results = client.results(search)
    
    for paper in tqdm(results, desc="Fetching papers"):
        # Extract authors
        authors_raw = ' | '.join([author.name for author in paper.authors])
        
        # Extract categories
        categories = ','.join(paper.categories)
        
        # Build paper data
        paper_info = {
            'arxiv_id': paper.get_short_id(),
            'title': paper.title,
            'authors_raw': authors_raw,
            'affiliations': '',  # ArXiv API doesn't always provide affiliations
            'summary': paper.summary.replace('\n', ' '),  # Clean up line breaks
            'published_date': paper.published.strftime('%Y-%m-%d'),
            'url': paper.pdf_url or paper.entry_id,
            'categories': categories
        }
        
        papers_data.append(paper_info)
        
except Exception as e:
    print(f"❌ Error during ArXiv search: {e}")
    raise

print(f"\n✅ Retrieved {len(papers_data)} papers")

In [ ]:
# Convert to DataFrame and inspect results
papers_df = pd.DataFrame(papers_data)

print(f"Papers DataFrame shape: {papers_df.shape}")
print(f"Columns: {list(papers_df.columns)}")

# Basic statistics
print(f"\nDate range: {papers_df['published_date'].min()} to {papers_df['published_date'].max()}")
print(f"Unique categories: {len(papers_df['categories'].str.split(',').explode().unique())}")

# Check for duplicates
duplicates = papers_df.duplicated(subset=['arxiv_id']).sum()
if duplicates > 0:
    print(f"⚠️  Found {duplicates} duplicate papers, removing...")
    papers_df = papers_df.drop_duplicates(subset=['arxiv_id'])
    print(f"Remaining papers: {len(papers_df)}")

# Show sample results
print("\n--- Sample papers ---")
for _, paper in papers_df.head(3).iterrows():
    print(f"Title: {paper['title'][:80]}...")
    print(f"Authors: {paper['authors_raw'][:60]}...")
    print(f"Date: {paper['published_date']}, Categories: {paper['categories']}")
    print()

In [ ]:
# Validate data quality
print("Validating paper data...")

# Check for missing critical fields
critical_fields = ['arxiv_id', 'title', 'authors_raw', 'summary']
for field in critical_fields:
    missing = papers_df[field].isna().sum()
    if missing > 0:
        print(f"⚠️  {missing} papers missing '{field}'")

# Check title lengths (should be reasonable)
short_titles = (papers_df['title'].str.len() < 10).sum()
if short_titles > 0:
    print(f"⚠️  {short_titles} papers have very short titles")

# Check author counts
single_authors = (papers_df['authors_raw'].str.count('\|') == 0).sum()
print(f"Single-author papers: {single_authors}")
print(f"Multi-author papers: {len(papers_df) - single_authors}")

# Category distribution
all_categories = papers_df['categories'].str.split(',').explode()
category_counts = all_categories.value_counts().head(10)
print(f"\nTop categories:")
for cat, count in category_counts.items():
    print(f"  {cat}: {count} papers")

print("\n✅ Validation complete")

In [ ]:
# Save papers to CSV
save_csv_checkpoint(papers_df, 'papers.csv')
print("✅ Papers saved to data/papers.csv")

# Final summary
print(f"\n📊 ArXiv Search Summary:")
print(f"- Search keywords: {len(search_keywords)}")
print(f"- Papers found: {len(papers_df)}")
print(f"- Date range: {papers_df['published_date'].min()} - {papers_df['published_date'].max()}")
print(f"- Categories covered: {len(all_categories.unique())}")
print(f"- Avg authors per paper: {papers_df['authors_raw'].str.count('|').mean() + 1:.1f}")

# Check if we hit the limit
if len(papers_df) >= config['arxiv']['max_results']:
    print(f"⚠️  Reached maximum results limit ({config['arxiv']['max_results']})")
    print("Consider increasing max_results in config or narrowing search criteria")